# SignalScope - Ensemble Combiner
Runs each frozen member over a labelled sample, then fits a logistic-regression
combiner on their output probabilities.

The combiner is fitted on a **CIFAKE train** sample and evaluated on a sample of
the **CIFAKE test split**, which the fit never touches.

In [ ]:
import os
import json
import glob
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models
import torchvision.transforms as T
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

# Keep inference tractable: three transformer members over every image is the
# expensive part, not the fitting.
N_FIT = 4000
N_TEST = 4000
BATCH = 32

## 1. Sample CIFAKE
Fit on CIFAKE train, score on the official CIFAKE test split - the same held-out
split the classifier run reports, so the numbers are directly comparable.

In [ ]:
CIFAKE_SLUG = "cifake-real-and-ai-generated-synthetic-images"
# Kaggle mounts attached datasets under either path depending on the notebook version.
CIFAKE_CANDIDATES = [f"/kaggle/input/datasets/birdy654/{CIFAKE_SLUG}", f"/kaggle/input/{CIFAKE_SLUG}"]
CIFAKE_ROOT = next((p for p in CIFAKE_CANDIDATES if os.path.isdir(os.path.join(p, "train", "REAL"))), None)
if CIFAKE_ROOT is None:
    raise SystemExit(f"CIFAKE not found under {CIFAKE_CANDIDATES}. Confirm the dataset is attached via Add Data.")
print("CIFAKE root:", CIFAKE_ROOT)


def list_split(split):
    """One row per image in a CIFAKE split. label: 0 = REAL, 1 = FAKE."""
    rows = []
    for cls, label in (("REAL", 0), ("FAKE", 1)):
        d = os.path.join(CIFAKE_ROOT, split, cls)
        rows += [(os.path.join(d, f), label) for f in sorted(os.listdir(d))]
    return pd.DataFrame(rows, columns=["abspath", "label"])


def balanced(df, n):
    k = min(n // 2, *df["label"].value_counts().tolist())
    return df.groupby("label").sample(k, random_state=SEED).sample(frac=1.0, random_state=SEED).reset_index(drop=True)


# CIFAKE's train and test folders are disjoint, so the fit and test sets never share an image.
fit_df = balanced(list_split("train"), N_FIT)
test_df = balanced(list_split("test"), N_TEST)

print(f"fit set={len(fit_df)} (fake {fit_df.label.sum()})   test set={len(test_df)} (fake {test_df.label.sum()})")

## 2. Frozen members
Three pretrained HuggingFace detectors plus the dual-stream checkpoint.
Nothing here is trained.

In [ ]:
class FrequencyBranch(nn.Module):
    def __init__(self, in_channels=1, feature_dim=128):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 32, 3, stride=2, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, stride=2, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, 3, stride=2, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))
        self.fc = nn.Linear(128 * 4 * 4, feature_dim)

    def extract_fft_spectrum(self, x):
        gray = 0.2989 * x[:, 0:1] + 0.5870 * x[:, 1:2] + 0.1140 * x[:, 2:3] if x.shape[1] == 3 else x
        log_spectrum = torch.log(torch.abs(torch.fft.fftshift(torch.fft.fft2(gray))) + 1e-8)
        flat = log_spectrum.view(log_spectrum.size(0), -1)
        mn = flat.min(dim=1, keepdim=True)[0].unsqueeze(-1).unsqueeze(-1)
        mx = flat.max(dim=1, keepdim=True)[0].unsqueeze(-1).unsqueeze(-1)
        return (log_spectrum - mn) / (mx - mn + 1e-8)

    def forward(self, x):
        f = self.extract_fft_spectrum(x)
        f = F.relu(self.bn1(self.conv1(f)))
        f = F.relu(self.bn2(self.conv2(f)))
        f = F.relu(self.bn3(self.conv3(f)))
        return F.relu(self.fc(torch.flatten(self.adaptive_pool(f), 1)))


class SignalScopeDualStreamModel(nn.Module):
    def __init__(self, spatial_backbone="resnet34", pretrained=False, dropout_rate=0.3):
        super().__init__()
        base = tv_models.resnet34(weights=None)
        num_spatial = base.fc.in_features
        base.fc = nn.Identity()
        self.spatial_stream = base
        self.freq_dim = 128
        self.frequency_stream = FrequencyBranch(1, self.freq_dim)
        self.classifier = nn.Sequential(
            nn.Linear(num_spatial + self.freq_dim, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Dropout(dropout_rate), nn.Linear(256, 64), nn.ReLU(),
            nn.Dropout(dropout_rate / 2), nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.classifier(torch.cat((self.spatial_stream(x), self.frequency_stream(x)), dim=1))


ckpt_paths = glob.glob("/kaggle/input/**/best_model.pt", recursive=True)
if not ckpt_paths:
    raise SystemExit("best_model.pt not found - add the training notebook's output as a data source.")
print("dual-stream checkpoint:", ckpt_paths[0])

ckpt = torch.load(ckpt_paths[0], map_location=DEVICE, weights_only=False)
dual = SignalScopeDualStreamModel().to(DEVICE)
dual.load_state_dict(ckpt["model_state_dict"])
dual.eval()
DUAL_TEMPERATURE = float(ckpt.get("temperature", 1.0))

NORM = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
if "image_size" in ckpt:
    dual_tf = T.Compose([T.Resize(ckpt.get("native_size", 200)), T.CenterCrop(ckpt["image_size"]), T.ToTensor(), NORM])
else:
    dual_tf = T.Compose([T.Resize((224, 224)), T.ToTensor(), NORM])
print("dual-stream transform:", dual_tf)

In [ ]:
from transformers import AutoImageProcessor, AutoModelForImageClassification

HF_MEMBERS = [
    "dima806/deepfake_vs_real_image_detection",
    "umm-maybe/AI-image-detector",
    "Organika/sdxl-detector",
]

AI_TOKENS = ("fake", "artificial", "synthetic", "generated", "ai-generated", "ai_generated")


def ai_class_index(model):
    for idx, label in model.config.id2label.items():
        if any(tok in label.lower() for tok in AI_TOKENS):
            return int(idx)
    return 1


hf_models = {}
for name in HF_MEMBERS:
    proc = AutoImageProcessor.from_pretrained(name)
    mdl = AutoModelForImageClassification.from_pretrained(name).to(DEVICE).eval()
    idx = ai_class_index(mdl)
    print(f"{name}: id2label={mdl.config.id2label} -> AI index {idx}")
    hf_models[name] = (proc, mdl, idx)

## 3. Member inference
Forward passes only - no gradients, no weight updates.

In [ ]:
@torch.no_grad()
def hf_probs(name, df):
    proc, mdl, idx = hf_models[name]
    out = []
    paths = df["abspath"].tolist()
    for start in range(0, len(paths), BATCH):
        imgs = [Image.open(p).convert("RGB") for p in paths[start:start + BATCH]]
        batch = proc(images=imgs, return_tensors="pt").to(DEVICE)
        logits = mdl(**batch).logits
        out.append(torch.softmax(logits, dim=1)[:, idx].float().cpu().numpy())
        if start % (BATCH * 20) == 0:
            print(f"   {name} {start}/{len(paths)}")
    return np.concatenate(out)


@torch.no_grad()
def dual_probs(df):
    out = []
    paths = df["abspath"].tolist()
    for start in range(0, len(paths), BATCH):
        imgs = torch.stack([dual_tf(Image.open(p).convert("RGB")) for p in paths[start:start + BATCH]]).to(DEVICE)
        logits = dual(imgs) / DUAL_TEMPERATURE
        out.append(torch.sigmoid(logits).squeeze(1).float().cpu().numpy())
    return np.concatenate(out)


def member_matrix(df, tag):
    print(f"\n--- scoring {tag} ({len(df)} images) ---")
    cols, names = [], []
    for name in HF_MEMBERS:
        cols.append(hf_probs(name, df))
        names.append(name)
    cols.append(dual_probs(df))
    names.append("signalscope_dual_stream")
    return np.column_stack(cols), names


X_fit, MEMBER_NAMES = member_matrix(fit_df, "fit set")
y_fit = fit_df["label"].values

X_test, _ = member_matrix(test_df, "CIFAKE test set")
y_test = test_df["label"].values

# A member whose labels are mapped backwards shows AUC < 0.5; flip it rather
# than letting the combiner fight the inverted signal.
for j, name in enumerate(MEMBER_NAMES):
    auc = roc_auc_score(y_fit, X_fit[:, j])
    if auc < 0.5:
        print(f"flipping inverted member {name} (fit AUC {auc:.3f})")
        X_fit[:, j] = 1.0 - X_fit[:, j]
        X_test[:, j] = 1.0 - X_test[:, j]

## 4. Fit the combiner and compare

In [ ]:
def report(name, y, probs, threshold=0.5):
    preds = (probs >= threshold).astype(int)
    cm = confusion_matrix(y, preds)
    tn, fp, fn, tp = cm.ravel()
    m = {
        "auc": float(roc_auc_score(y, probs)),
        "macro_f1": float(f1_score(y, preds, average="macro")),
        "accuracy": float((tp + tn) / cm.sum()),
        "fpr": float(fp / (fp + tn)) if (fp + tn) else 0.0,
        "confusion": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    }
    print(f"{name:38s} AUC={m['auc']:.4f}  F1={m['macro_f1']:.4f}  acc={m['accuracy']*100:.1f}%  FPR={m['fpr']*100:.1f}%")
    return m


print("\n=== individual members on the CIFAKE test split ===")
member_metrics = {n: report(n, y_test, X_test[:, j]) for j, n in enumerate(MEMBER_NAMES)}

combiner = LogisticRegression(max_iter=1000, C=1.0)
combiner.fit(X_fit, y_fit)

p_test = combiner.predict_proba(X_test)[:, 1]
p_fit = combiner.predict_proba(X_fit)[:, 1]

print("\n=== ensemble ===")
m_fit = report("learned ensemble (fit set)", y_fit, p_fit)
m_test = report("learned ensemble (CIFAKE test split)", y_test, p_test)

# Current hardcoded soft-voting weights, for an honest side-by-side.
HARDCODED = np.array([0.25, 0.22, 0.20, 0.18])
hard_w = HARDCODED[: len(MEMBER_NAMES)] / HARDCODED[: len(MEMBER_NAMES)].sum()
m_hard = report("hardcoded soft-vote (CIFAKE test split)", y_test, X_test @ hard_w)

LOW_FPR_THRESHOLD = float(np.quantile(p_fit[y_fit == 0], 0.95))
print(f"\nthreshold for ~5% FPR on fit set: {LOW_FPR_THRESHOLD:.4f}")
m_test_lowfpr = report("learned ensemble @ 5% FPR", y_test, p_test, LOW_FPR_THRESHOLD)

print("\nlearned weights:")
for n, w in zip(MEMBER_NAMES, combiner.coef_[0]):
    print(f"  {n:40s} {w:+.4f}")
print(f"  {'intercept':40s} {combiner.intercept_[0]:+.4f}")

## 5. Save

In [ ]:
ensemble_spec = {
    "dataset": f"CIFAKE (birdy654/{CIFAKE_SLUG})",
    "members": MEMBER_NAMES,
    "coefficients": combiner.coef_[0].tolist(),
    "intercept": float(combiner.intercept_[0]),
    "low_fpr_threshold": LOW_FPR_THRESHOLD,
    "dual_stream_temperature": DUAL_TEMPERATURE,
    "fit_size": int(len(fit_df)),
    "test_size": int(len(test_df)),
    "metrics": {
        "individual_members_test": member_metrics,
        "ensemble_fit": m_fit,
        "ensemble_test": m_test,
        "ensemble_test_at_5pct_fpr": m_test_lowfpr,
        "hardcoded_softvote_test": m_hard,
    },
}

with open("/kaggle/working/ensemble_weights.json", "w") as f:
    json.dump(ensemble_spec, f, indent=2)

print("\nSaved /kaggle/working/ensemble_weights.json")